# Coffee17 Master Screening — Analysis Package Only

Use this notebook **only after the master screening has already finished** and its Kaggle saved output is attached as an Input.

This notebook does **not train**, does **not require GPU**, and never loads model checkpoints. It only extracts compact result artifacts into one ZIP for external analysis.


In [ ]:
NOTEBOOK_BUILD = "MASTER-SCREENING-ANALYSIS-ONLY-V1"
import json, shutil, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
OUTPUT_ZIP = WORK / "coffee17-master-screening-analysis-package.zip"
STAGE = WORK / "_coffee17_master_analysis_only_stage"

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("TRAINING: DISABLED")
print("GPU: NOT REQUIRED")

shutil.rmtree(STAGE, ignore_errors=True)
STAGE.mkdir(parents=True, exist_ok=True)
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

# 1) Prefer an already-created compact package from the attached saved output.
existing_packages = sorted(
    p for p in INPUT.rglob("coffee17-master-screening-analysis-package.zip")
    if p.is_file()
)
if len(existing_packages) > 1:
    raise RuntimeError(
        "Lebih dari satu compact analysis package ditemukan: "
        + str([str(p) for p in existing_packages])
    )

if existing_packages:
    source = existing_packages[0]
    shutil.copy2(source, OUTPUT_ZIP)
    print("REUSED EXISTING ANALYSIS PACKAGE:", source)
else:
    # 2) Otherwise reconstruct a compact package from the completed master project.
    project_roots = sorted(
        p for p in INPUT.rglob("coffee17-master-representation-screening-v1")
        if p.is_dir()
    )
    if not project_roots:
        raise RuntimeError(
            "Output master screening tidak ditemukan. Attach Saved Version yang "
            "berisi folder coffee17-master-representation-screening-v1."
        )
    if len(project_roots) > 1:
        # Prefer a root that already contains the final master summary.
        with_summary = [
            p for p in project_roots
            if (p / "analysis" / "master_screening_v1.json").is_file()
        ]
        if len(with_summary) == 1:
            project = with_summary[0]
        else:
            raise RuntimeError(
                "Lebih dari satu project master ditemukan: "
                + str([str(p) for p in project_roots])
            )
    else:
        project = project_roots[0]

    summary = project / "analysis" / "master_screening_v1.json"
    if not summary.is_file():
        raise RuntimeError(
            "Master summary tidak ditemukan. Output yang di-attach tampaknya "
            "belum menyelesaikan seluruh screening."
        )

    # Validate that all five fold results exist before packaging.
    fold_results = []
    for fold in range(1, 6):
        path = project / "experiments" / f"fold_{fold}" / "seed42" / "fold_result.json"
        if not path.is_file():
            raise RuntimeError(f"Fold result hilang: {path}")
        fold_results.append(path)

    def copy_file(src: Path, rel: Path):
        dst = STAGE / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)

    copy_file(summary, Path("analysis/master_screening_v1.json"))

    # Compact per-fold/per-arm evidence only; NEVER copy checkpoints.
    for fold in range(1, 6):
        fold_root = project / "experiments" / f"fold_{fold}" / "seed42"
        dst_fold = Path("experiments") / f"fold_{fold}" / "seed42"
        copy_file(fold_root / "fold_result.json", dst_fold / "fold_result.json")

        for arm in sorted(p for p in fold_root.iterdir() if p.is_dir()):
            dst_arm = dst_fold / arm.name
            for name in ("run_contract.json", "run_config.yaml", "history.json"):
                src = arm / name
                if src.is_file():
                    copy_file(src, dst_arm / name)

            validation = arm / "validation"
            if validation.is_dir():
                for src in sorted(p for p in validation.rglob("*") if p.is_file()):
                    copy_file(src, dst_arm / "validation" / src.relative_to(validation))

    # Explicitly record what is and is not included.
    files = [
        {
            "path": str(p.relative_to(STAGE)),
            "bytes": p.stat().st_size,
        }
        for p in sorted(q for q in STAGE.rglob("*") if q.is_file())
    ]
    (STAGE / "PACKAGE_MANIFEST.json").write_text(
        json.dumps(
            {
                "format": "coffee17.master_screening.analysis_only_package.v1",
                "training_performed": False,
                "checkpoints_included": False,
                "source_project": str(project),
                "files": files,
            },
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )

    shutil.make_archive(
        str(OUTPUT_ZIP.with_suffix("")),
        "zip",
        root_dir=STAGE,
    )
    print("BUILT ANALYSIS PACKAGE FROM:", project)

if not OUTPUT_ZIP.is_file():
    raise RuntimeError("Analysis ZIP gagal dibuat.")

with zipfile.ZipFile(OUTPUT_ZIP) as zf:
    names = zf.namelist()
    checkpoint_names = [n for n in names if n.endswith(".pt")]
    if checkpoint_names:
        raise RuntimeError(
            "Compact package tidak boleh mengandung checkpoint: "
            + str(checkpoint_names[:10])
        )
    if not any(n.endswith("master_screening_v1.json") for n in names):
        raise RuntimeError("Master summary tidak ditemukan di analysis package.")

print("READY:", OUTPUT_ZIP)
print("SIZE_MB:", round(OUTPUT_ZIP.stat().st_size / 1024 / 1024, 2))
print("FILES:", len(names))
print("CHECKPOINTS_INCLUDED: 0")
print("UPLOAD THIS ZIP TO CHAT FOR ANALYSIS.")
